# E12 - EfficientNet-B3, random oversampling, 300px

| | |
|---|---|
| Architecture | `efficientnet_b3` |
| Imbalance strategy | `oversampling` |
| Input resolution | 300x300 |
| Group | EfficientNet-B3 ablation |
| Optimiser | Adam, lr 1e-3, betas (0.9, 0.999) |
| Batch size | 32 |
| Schedule | max 50 epochs, early stopping on val loss, patience 10 |
| Seed | 42 |

B3 at 300 with oversampling, paired against E9. Last training run in the
set. Once this finishes, move on to notebook 20.

Training is checkpointed after every epoch. If the pod drops, re-run the training cell
and it picks up from the last completed epoch rather than starting over.

In [5]:
import json, sys, time
from pathlib import Path

# Works whether the kernel starts in notebooks/ or at the repo root.
ROOT = Path.cwd()
while not (ROOT / "src" / "ham10000").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# /workspace is a MooseFS network volume and per-file reads over FUSE were the
# bottleneck, not the GPU: 45 img/s there against 241 img/s from RAM. Notebook 01
# stages a copy into /dev/shm. Fall back to the on-disk copy if it is missing,
# since tmpfs does not survive a pod restart.
FAST = Path("/dev/shm/ham_data")
DATA = FAST if (FAST / "HAM10000_metadata.csv").exists() else ROOT / "data"
RUNS = ROOT / "runs"
print("repo:   ", ROOT)
print("data:   ", DATA, "(RAM)" if str(DATA).startswith("/dev/shm") else "(disk, slower)")

repo:    /workspace/ham10000-cnn-comparison
data:    /dev/shm/ham_data (RAM)


In [6]:
from ham10000.constants import EXPERIMENTS, experiment_label
from ham10000.train import TrainConfig, train_experiment

EXP = "E12"
print(experiment_label(EXP))
print(EXPERIMENTS[EXP])

E12 EfficientNet-B3 @300 / Random oversampling
{'architecture': 'efficientnet_b3', 'strategy': 'oversampling', 'image_size': 300}


## Preflight

Confirms the GPU, the data and the shared split before anything expensive starts.

In [7]:
import torch
from ham10000.split import load_or_create_split, split_hash

assert torch.cuda.is_available(), "No GPU. Stop here."
print("gpu :", torch.cuda.get_device_name(0))

assert (DATA / "HAM10000_metadata.csv").exists(), "Run notebook 01 first."
df, lesion_splits = load_or_create_split(DATA / "HAM10000_metadata.csv",
                                         RUNS / "splits" / "seed42_lesion_stratified.json")
print("split hash :", split_hash(lesion_splits))
print(df.split.value_counts().to_dict())

gpu : NVIDIA RTX A4500
split hash : f35ac1de18678182
{'train': 8012, 'test': 1024, 'val': 979}


## Train

Expect this to take a while. At 300px each epoch handles about 1.8x the pixels of the 224px runs, so budget accordingly.

The cell is safe to re-run: if `test_metrics.json` already exists it returns the cached
result instead of retraining. Pass `force_rerun=True` if you genuinely want it redone.

In [ ]:
config = TrainConfig(
    experiment_id=EXP,
    data_root=str(DATA),
    output_dir=str(RUNS / EXP),
    seed=42,
    batch_size=32,
    lr=1e-3,
    max_epochs=50,
    patience=10,
    num_workers=16,
)

start = time.time()
result = train_experiment(config)
print(f"\nwall clock: {(time.time() - start) / 60:.1f} min")

vram: 20.8 GB free, ~10.2 GB needed
Epoch 1/50 | train loss 0.7456 acc 0.7370 | val loss 0.8973 macro-F1 0.6237
Epoch 2/50 | train loss 0.4427 acc 0.8434 | val loss 1.1244 macro-F1 0.6259
Epoch 3/50 | train loss 0.3420 acc 0.8714 | val loss 0.5957 macro-F1 0.7111
Epoch 4/50 | train loss 0.2809 acc 0.8998 | val loss 0.6746 macro-F1 0.6733
Epoch 5/50 | train loss 0.2356 acc 0.9114 | val loss 0.6770 macro-F1 0.7232
Epoch 6/50 | train loss 0.2352 acc 0.9175 | val loss 0.5397 macro-F1 0.7181
Epoch 7/50 | train loss 0.2060 acc 0.9282 | val loss 0.6387 macro-F1 0.6876
Epoch 8/50 | train loss 0.1979 acc 0.9281 | val loss 0.5118 macro-F1 0.7424
Epoch 9/50 | train loss 0.1796 acc 0.9333 | val loss 0.6697 macro-F1 0.7194
Epoch 10/50 | train loss 0.1579 acc 0.9432 | val loss 0.6320 macro-F1 0.6913
Epoch 11/50 | train loss 0.1699 acc 0.9422 | val loss 0.7761 macro-F1 0.7050
Epoch 12/50 | train loss 0.1607 acc 0.9415 | val loss 0.7066 macro-F1 0.7297
Epoch 13/50 | train loss 0.1549 acc 0.9451 | val 

## Results

In [ ]:
import pandas as pd
from ham10000.constants import CLASS_NAMES

m = result["test_metrics"]
report = m["classification_report"]

print(f"accuracy          {m['accuracy']:.4f}")
print(f"balanced accuracy {m['balanced_accuracy']:.4f}")
print(f"macro F1          {m['macro_f1']:.4f}")
print(f"macro AUC (OvR)   {m['macro_auc_ovr']:.4f}\n")

table = pd.DataFrame([{
    "class": c,
    "precision": report[str(i)]["precision"],
    "recall": report[str(i)]["recall"],
    "f1": report[str(i)]["f1-score"],
    "support": int(report[str(i)]["support"]),
} for i, c in enumerate(CLASS_NAMES)]).set_index("class").round(4)
print(table)

rq2 = (report[str(CLASS_NAMES.index("mel"))]["recall"]
       + report[str(CLASS_NAMES.index("bcc"))]["recall"]) / 2
print(f"\nRQ2 criterion, mean(recall MEL, recall BCC) = {rq2:.4f}")

### Training curve

Val loss is what early stopping watches. Val macro F1 is what actually matters, and the
two do not always move together.

In [ ]:
import matplotlib.pyplot as plt

h = pd.read_csv(RUNS / EXP / "history.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h.epoch, h.train_loss, label="train")
axes[0].plot(h.epoch, h.val_loss, label="val")
best = h.loc[h.val_loss.idxmin()]
axes[0].axvline(best.epoch, ls="--", c="grey", lw=1)
axes[0].annotate(f"best epoch {int(best.epoch)}", (best.epoch, best.val_loss),
                 textcoords="offset points", xytext=(8, 10), fontsize=8)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title(f"{EXP} loss")

axes[1].plot(h.epoch, h.val_macro_f1, color="#D54407", label="val macro F1")
axes[1].plot(h.epoch, h.val_balanced_accuracy, color="#2E7D32", label="val balanced acc")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title(f"{EXP} validation metrics")
plt.show()

print(f"stopped after {len(h)} epochs, best val loss {h.val_loss.min():.4f} at epoch {int(best.epoch)}")

### Confusion matrix

In [ ]:
import numpy as np

cm = np.array(m["confusion_matrix"], dtype=float)
norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(norm, cmap="Oranges", vmin=0, vmax=1)
for i in range(7):
    for j in range(7):
        if cm[i, j]:
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if norm[i, j] > 0.55 else "black")
ax.set_xticks(range(7), [c.upper() for c in CLASS_NAMES], rotation=45)
ax.set_yticks(range(7), [c.upper() for c in CLASS_NAMES])
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"{EXP} confusion matrix (counts, shaded by row-normalised recall)")
ax.grid(False)
plt.show()

mel, nv = CLASS_NAMES.index("mel"), CLASS_NAMES.index("nv")
print(f"melanomas called NV: {int(cm[mel, nv])} of {int(cm[mel].sum())}")
print("This is the clinically costly error: a cancer sent home as a benign mole.")

### Files written

These five are what the analysis notebooks read. Nothing later re-derives them.

In [ ]:
for f in sorted((RUNS / EXP).iterdir()):
    print(f"{f.name:<24} {f.stat().st_size/1e6:>8.2f} MB")